# 09-Fault tolerance：超时/重试/错误处理/优雅停止与恢复

## 容错：让图在不稳定世界里继续跑

Retries · Timeouts · Error handler · Drain · 第九课 · 约 40 分钟

### 目录

- 回顾：图越来越像“真实系统”
- 一、为什么必须考虑容错？
- 二、重试：把“偶发失败”当作常态
- 三、超时：避免一个节点拖垮整张图
- 四、错误处理：重试耗尽后的兜底分支（Command goto）
- 五、把稳定性策略设成默认：set_node_defaults
- 六、优雅停止与恢复（RunControl.request_drain）
- 七、动手跑一下
- 八、常见坑速查
- 九、总结
- 📖 参考（官方）


### 回顾：图越来越像“真实系统”

前面几节我们解决的是“能表达工作流”（条件路由、HIL、Streaming、Store、Subgraph）。

当你的图开始接入外部系统（模型、支付、数据库、HTTP API），你会发现最常见的问题不是“写错了”，而是：

- **偶发失败**：网络抖一下、上游 502、数据库临时锁表。
- **慢**：某次调用突然慢很多，拖住整个流程。
- **失败后怎么继续**：是直接让整张图报错退出？还是降级？还是记录失败并结束？
- **要停机/发布**：正在跑的流程怎么“安全停下”，之后还能恢复？

这一节的核心直觉和第 04/05 节很像：

- Checkpointer 解决“记得住”。
- Human-in-the-Loop 解决“能停/能继续”。
- **Fault tolerance 解决“遇到不稳定时也能按你的规则继续/结束/恢复”。**


In [1]:
import random
import time
from typing import Annotated

import operator
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END


class FTState(TypedDict):
    """Fault-tolerance demo state (最小可跑)。"""

    request_id: str
    logs: Annotated[list[str], operator.add]
    result: str


def now_ms() -> int:
    return int(time.time() * 1000)


def print_title(title: str):
    print("\n" + "=" * 20)
    print(title)
    print("=" * 20)


def print_logs(state: FTState):
    print("--- logs ---")
    for line in state["logs"]:
        print(line)
    print("-----------")


init_state: FTState = {
    "request_id": "req-001",
    "logs": [],
    "result": "",
}

print("=== init_state ===")
print(init_state)


=== init_state ===
{'request_id': 'req-001', 'logs': [], 'result': ''}


---

### 一、为什么必须考虑容错？

把“外部不稳定”想象成快递物流：

- 偶尔丢包/超时（请求失败）
- 偶尔堵车（请求变慢）
- 偶尔系统维护（你得停机）

你真正想要的不是“永不失败”，而是：

- **失败时按规则重试**（外部抖动就别一棒子打死）
- **太慢就及时止损**（不能让一个节点拖垮整张图）
- **实在不行就走兜底**（给一个可解释的结果/状态）
- **要停机时能安全停下并恢复**（不是粗暴 kill）

接下来我们按“从简单到工程”的顺序学：重试 → 超时 → 兜底 → 默认策略 → 优雅停止/恢复。

In [2]:
from langgraph.runtime import Runtime
from langgraph.types import RetryPolicy


class TransientAPIError(Exception):
    """模拟：可重试的瞬时错误（比如 502 / 网络闪断）。"""


def call_external_api(state: FTState, runtime: Runtime) -> dict:
    """节点：用 node_attempt 制造“必然重试”的可观察效果。"""

    attempt = runtime.execution_info.node_attempt
    ts = now_ms()

    # 前两次必失败，第 3 次成功：保证你能在一次运行里看到 retry 的意义
    if attempt < 3:
        raise TransientAPIError(f"[{ts}] transient failure (attempt={attempt})")

    return {
        "logs": [f"[{ts}] call_external_api: success (attempt={attempt})"],
        "result": f"OK({state['request_id']})",
    }


builder_retry = StateGraph(FTState)
builder_retry.add_node(
    "调用外部API",
    call_external_api,
    retry_policy=RetryPolicy(
        max_attempts=5,
        retry_on=TransientAPIError,  # 只重试我们认为“值得重试”的异常
    ),
)
builder_retry.add_edge(START, "调用外部API")
builder_retry.add_edge("调用外部API", END)

graph_retry = builder_retry.compile()

---

### 二、重试：把“偶发失败”当作常态

把问题说得更具体一点：

- 你有一个节点负责调用外部 API（模型/数据库/HTTP）。
- 它 **偶尔** 会失败。
- 你不想因为一次偶发失败就让整张图挂掉。

这时最直接的策略就是：**失败后再试几次**。

重试的关键不是“多试几次”，而是三件事：

- **最多试几次**（`max_attempts`）
- **哪些异常才值得重试**（`retry_on`）
- **第几次尝试**（用 `runtime.execution_info.node_attempt` 做降级/切备用方案）

> **❓ 检查理解 ①**
>
> 下面哪一种错误更应该“重试”？
>
> - A. `ValueError`（比如你自己解析 JSON 写错了字段名）
> - B. `TransientAPIError`（比如上游临时 502 / 网络闪断）
> - C. `SyntaxError`（比如你写的代码本身语法就不对）
>
> **✅ 答案：B**
>
> 解释：B 属于“外部瞬时不稳定”，重试可能成功；A/C 更像是你代码或输入的问题，重试只是重复失败，反而掩盖 bug。


In [3]:
import asyncio


async def slow_node(state: FTState) -> dict:
    ts = now_ms()
    # 模拟：外部系统偶尔非常慢
    await asyncio.sleep(0.2)
    return {
        "logs": [f"[{ts}] slow_node: finished after sleep"],
        "result": "SLOW_OK",
    }


builder_timeout = StateGraph(FTState)
builder_timeout.add_node(
    "慢调用",
    slow_node,
    timeout=0.05,  # 50ms：故意设置得很小，保证触发超时
    retry_policy=RetryPolicy(max_attempts=2),
)
builder_timeout.add_edge(START, "慢调用")
builder_timeout.add_edge("慢调用", END)

graph_timeout = builder_timeout.compile()

---

### 三、超时：避免一个节点拖垮整张图

重试解决的是“偶发失败”，但还有另一类更隐蔽的问题：**卡住/太慢**。

- 如果一个节点调用外部系统，有时候会突然慢到几分钟。
- 如果不加限制，整张图就会像“排队堵死”。

LangGraph 支持给节点设置 `timeout=`：超过上限会抛出 `NodeTimeoutError`。

超时这一块有两个关键点（很多人第一次都会踩）：

- **timeout 只对 async 节点生效**
- 你要用 **异步执行** 才能触发真正的超时（notebook 里通常是 `await graph.ainvoke(...)`）

> **❓ 检查理解 ②**
>
> 为什么 `timeout` 强调“只对 async 节点生效”？
>
> - A. 因为 async 节点更快，sync 节点太慢
> - B. 因为超时/取消依赖 asyncio 的任务管理；同步执行无法在进程内安全中断
> - C. 因为 `timeout` 只对 LLM 节点有效，对普通函数无效
>
> **✅ 答案：B**
>
> 解释：超时本质上是“取消任务”。async 节点运行在 asyncio 任务里可以被取消；同步代码一般无法在不杀线程/进程的情况下被安全中断。
>
> 经验做法：如果你必须调用阻塞库（同步 HTTP / 同步数据库），把它包进 `asyncio.to_thread(...)`，再把节点写成 `async def`。

---

### 四、错误处理：重试耗尽后的兜底分支（Command goto）

有时候，“重试”也救不了：

- 外部系统长期不可用
- 或者错误就是不可重试的（比如鉴权失败、请求参数不合法）

这时你需要的是 **兜底逻辑**：

- 记录失败原因（写入 state 的 logs/status）
- 走到一个“失败收尾”节点（比如发告警、落库、返回友好提示）

LangGraph 用 `error_handler=` 来做这件事：

- 节点抛异常
- retry_policy 决定不重试/或重试耗尽
- **error_handler 接管**，并可用 `Command(goto=...)` 跳转到兜底分支


In [4]:
from langgraph.types import Command
from langgraph.errors import NodeError


def always_fail(state: FTState) -> dict:
    ts = now_ms()
    raise RuntimeError(f"[{ts}] upstream hard failure")


def fallback_finalize(state: FTState) -> dict:
    ts = now_ms()
    return {
        "logs": [f"[{ts}] fallback_finalize: degraded path"],
        "result": "DEGRADED_OK",
    }


def failure_handler(state: FTState, error: NodeError) -> Command:
    ts = now_ms()
    return Command(
        update={
            "logs": [
                f"[{ts}] error_handler: captured error_type={type(error).__name__}",
                f"[{ts}] error_handler: captured error={str(error)[:120]}",
            ]
        },
        goto="失败兜底",
    )


builder_handler = StateGraph(FTState)
builder_handler.add_node(
    "主调用",
    always_fail,
    retry_policy=RetryPolicy(
        max_attempts=2,
        retry_on=TransientAPIError,  # 只重试瞬时错误；这里抛 RuntimeError，所以不会重试
    ),
    error_handler=failure_handler,
)
builder_handler.add_node("失败兜底", fallback_finalize)

builder_handler.add_edge(START, "主调用")
builder_handler.add_edge("主调用", END)  # 正常成功时（本例不会）
builder_handler.add_edge("失败兜底", END)

graph_handler = builder_handler.compile()

---

### 五、把稳定性策略设成默认：set_node_defaults

当你开始写“工程级”图时，几乎每个节点都会想要：

- 一个默认的 `retry_policy`
- 一个默认的 `timeout`
- 一个默认的 `error_handler`（比如统一打日志/落库/标记流程失败）

如果你每个 `add_node(...)` 都重复写一遍，既啰嗦也容易漏。

这时用 `set_node_defaults(...)`：把“稳定性策略”像项目配置一样集中管理。


In [5]:
def step_a(state: FTState) -> dict:
    ts = now_ms()
    return {"logs": [f"[{ts}] step_a: ok"]}


def step_b(state: FTState) -> dict:
    ts = now_ms()
    # 模拟：偶发失败
    if random.random() < 0.3:
        raise TransientAPIError("b transient")
    return {"logs": [f"[{ts}] step_b: ok"], "result": "PIPELINE_OK"}


def default_handler(state: FTState, error: NodeError):
    ts = now_ms()
    return Command(
        update={"logs": [f"[{ts}] default_handler: {type(error).__name__} {str(error)[:80]}"]},
        goto=END,  # 最简单兜底：记录一下就结束
    )


builder_defaults = (
    StateGraph(FTState)
    .set_node_defaults(
        retry_policy=RetryPolicy(max_attempts=3, retry_on=TransientAPIError),
        error_handler=default_handler,
        # timeout 这里不演示默认值（timeout 是 async-only）
    )
)

builder_defaults.add_node("A", step_a)
builder_defaults.add_node("B", step_b)
builder_defaults.add_edge(START, "A")
builder_defaults.add_edge("A", "B")
builder_defaults.add_edge("B", END)

graph_defaults = builder_defaults.compile()

---

### 六、优雅停止与恢复（RunControl.request_drain）

有些时候你想“停下来”，不是因为报错，而是因为：

- 你收到了 SIGTERM（容器要被重启）
- 你想做发布/扩容，希望正在跑的图 **在一个安全边界停下**
- 你希望稍后能继续跑（配合 checkpointer）

LangGraph 的思路是：

- **不要粗暴 kill**（那会让 state 不完整/副作用不可控）
- 而是请求“排空（drain）”：**在下一个 superstep 边界停下**

这需要 `RunControl`：

- 在另一个线程/信号处理器里调用 `control.request_drain(reason)`
- 图在合适的边界抛出 `GraphDrained(reason)`
- 只要你配置了 checkpointer，就能在之后用同一个 `thread_id` 继续执行

> 注意：`request_drain()` **不会** 立刻取消正在执行的 asyncio 任务/线程。它的目标是“安全边界停”，不是“立刻停”。


In [6]:
import threading

from langgraph.runtime import RunControl
from langgraph.errors import GraphDrained
from langgraph.checkpoint.memory import InMemorySaver


def step1(state: FTState) -> dict:
    ts = now_ms()
    time.sleep(0.05)
    return {"logs": [f"[{ts}] step1: ok"]}


def step2(state: FTState) -> dict:
    ts = now_ms()
    time.sleep(0.05)
    return {"logs": [f"[{ts}] step2: ok"], "result": "DONE"}


checkpointer = InMemorySaver()

builder_drain = StateGraph(FTState)
builder_drain.add_node("步骤1", step1)
builder_drain.add_node("步骤2", step2)
builder_drain.add_edge(START, "步骤1")
builder_drain.add_edge("步骤1", "步骤2")
builder_drain.add_edge("步骤2", END)

graph_drain = builder_drain.compile(checkpointer=checkpointer)

control = RunControl()
config = {"configurable": {"thread_id": "drain-001"}}


def request_drain_soon():
    # 在一个合适窗口请求排空（更稳定复现 GraphDrained）
    time.sleep(0.03)
    control.request_drain("demo-stop")

---

### 七、动手跑一下

下面把前面的四块机制串起来，做成一组“可观察”的最小实验：

- Demo 1：重试（必然重试 2 次）
- Demo 2：超时（用 `await graph.ainvoke` 触发 `NodeTimeoutError`）
- Demo 3：错误兜底（error_handler + Command(goto)）
- Demo 4：默认策略（set_node_defaults）
- Demo 5：优雅停止与恢复（drain + 同 thread_id 恢复）


In [7]:
# 为了让输出更稳定可复现
random.seed(7)

print_title("Demo 1: retry_policy（必然重试两次）")
out = graph_retry.invoke({**init_state, "logs": [], "result": ""})
print("result:", out["result"])
print_logs(out)

print_title("Demo 2: timeout（async-only）")
try:
    await graph_timeout.ainvoke({**init_state, "logs": [], "result": ""})
except Exception as e:
    print("caught:", type(e).__name__)
    print(str(e))

print_title("Demo 3: error_handler + Command(goto)")
out = graph_handler.invoke({**init_state, "logs": [], "result": ""})
print("result:", out["result"])
print_logs(out)

print_title("Demo 4: set_node_defaults")
out = graph_defaults.invoke({**init_state, "logs": [], "result": ""})
print("result:", out["result"])
print_logs(out)

print_title("Demo 5: graceful drain")
# 让线程去请求 drain
threading.Thread(target=request_drain_soon).start()
try:
    out = graph_drain.invoke({**init_state, "logs": [], "result": ""}, config=config, control=control)
    print("finished normally, result:", out.get("result"))
except GraphDrained as e:
    print("GraphDrained reason:", e.reason)

print_title("Resume after drain")
out2 = graph_drain.invoke(None, config=config)
print("result:", out2["result"])
print_logs(out2)


Demo 1: retry_policy（必然重试两次）


result: OK(req-001)
--- logs ---
[1786085042242] call_external_api: success (attempt=3)
-----------

Demo 2: timeout（async-only）
caught: NodeTimeoutError
Node '慢调用' exceeded its run timeout of 0.050s (elapsed: 0.051s).

Demo 3: error_handler + Command(goto)
result: DEGRADED_OK
--- logs ---
[1786085043290] error_handler: captured error_type=NodeError
[1786085043290] error_handler: captured error=NodeError(node='主调用', error=RuntimeError('[1786085043288] upstream hard failure'))
[1786085043292] fallback_finalize: degraded path
-----------

Demo 4: set_node_defaults
result: PIPELINE_OK
--- logs ---
[1786085043296] step_a: ok
[1786085043297] step_b: ok
-----------

Demo 5: graceful drain
GraphDrained reason: demo-stop

Resume after drain
result: DONE
--- logs ---
[1786085043304] step1: ok
[1786085043359] step2: ok
-----------


### 八、常见坑速查

| 症状 | 原因 | 解法 |
| --- | --- | --- |
| `timeout=` 没生效 / 报错说只支持 async | 你在同步执行里跑（`invoke`），或节点不是 `async def` | 节点写成 `async def`，并用 `await graph.ainvoke(...)` |
| 逻辑 bug 也在重试 | `retry_on` 太宽，什么异常都重试 | 只重试“瞬时可恢复”的异常（网络、5xx、临时锁等） |
| 超时后你以为写入会保留 | 超时 attempt 的写入会被清空 | 把关键写入放在能完成的节点里；或用更合适的 timeout / heartbeat |
| drain 请求了但没立刻停 | drain 是“在 superstep 边界停” | 需要硬上限就配合 timeout/任务取消；drain 只保证安全边界 |
| 以为 `interrupt()` 也会走 error_handler | `interrupt()` 走 bubble-up 暂停语义 | HIL 用 interrupt/resume；错误兜底用 error_handler |

### 九、总结

| 机制 | 一句话解释 | 你应该记住什么 |
| --- | --- | --- |
| `RetryPolicy` | 节点失败后按规则自动重试 | 只重试你认为“瞬时可恢复”的异常；用 `node_attempt` 做降级 |
| `timeout=` / `TimeoutPolicy` | 单次尝试的时间上限，超时抛 `NodeTimeoutError` | **async-only + async 执行**；超时 attempt 写入会被清空 |
| `error_handler=` | 重试耗尽后跑兜底逻辑，可用 `Command(goto=...)` 走失败分支 | 把“失败时怎么结束/怎么补偿”写成一条明确路径 |
| `set_node_defaults` | 给整张图设置默认的重试/兜底等策略 | 少写重复参数；项目级统一风格，节点可局部覆盖 |
| `RunControl.request_drain` | 请求图在安全边界停止并可恢复 | 停机/发布时很关键；配合 checkpointer + 同 thread_id 恢复 |

> **一句话总结**  
> 重试解决“偶发失败”，超时解决“太慢卡死”，兜底解决“失败后怎么收尾”，drain 解决“要停机时怎么安全停并恢复”。

📖 参考：

- [LangGraph 官方文档 - Fault tolerance](https://docs.langchain.com/oss/python/langgraph/fault-tolerance)
- [LangGraph 官方文档 - Graph API（retry_policy / timeout / error_handler / set_node_defaults）](https://docs.langchain.com/oss/python/langgraph/graph-api)
